# Monitoring long training runs

Reads every training log under a run directory (e.g. `cpbench_runs_long`, copied back from the cluster with `rsync -a --include='*/' --include='log.jsonl' --include='learner.json' --exclude='*' cluster:path/cpbench_runs_long/ cpbench_runs_long/`) and plots, per dataset and learner (mean over seeds, shaded min–max):

- the training loss (log–log: slow power-law decay shows as a straight line);
- the periodic validation evaluation: supervised probes (linear on windows, MLP on window pairs; diagnostics), unsupervised accuracy of each detector in `eval_detectors` at sample level, and the effective rank of the features.

The tables report the last evaluation, the peak of each curve and where it occurred (a peak well before the end means the objective drifts away from regime structure), and how much the loss still fell over the last fifth of training (a convergence indicator).

Logs from runs before the extended evaluation only have `probe` and `gmm_acc`. Reading logs costs nothing; the optional last section evaluates kept checkpoints on the test split (needs the model files).

In [ ]:
import os, json, glob, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from cpbench import Benchmark, make_learner
from cpbench.detectors import make_detector
from cpbench.eval.scoring import score
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "font.size": 8})
pd.set_option("display.width", 220)
warnings.filterwarnings("ignore", message="FastICA did not converge")

RUNS = os.environ.get("CPBENCH_RUNS", "cpbench_runs_long")     # ✏️ run directory to monitor

In [ ]:
def load_logs(root):
    train, evals = [], []
    for p in glob.glob(os.path.join(root, "*", "*", "*", "s*", "log.jsonl")):
        dset, learner, tag, seed = p.split(os.sep)[-5:-1]
        meta = dict(dataset=dset, learner=learner, tag=tag, seed=int(seed[1:]), run=os.path.dirname(p))
        for line in open(p):
            r = json.loads(line)
            (train if r["kind"] == "train" else evals if r["kind"] == "eval" else []).append({**meta, **r})
    return pd.DataFrame(train), pd.DataFrame(evals)

train, evals = load_logs(RUNS)
print(f"{RUNS}: {train.run.nunique() if len(train) else 0} runs, datasets: {sorted(train.dataset.unique()) if len(train) else []}")
if len(train):
    display(train.groupby(["dataset", "learner"]).agg(seeds=("seed", "nunique"), last_step=("step", "max")))

## Curves

In [ ]:
METRICS = ["probe", "probe_mlp_pair", "gmm_acc", "hmm_acc", "arhmm_acc", "temporal_excess_acc", "eff_rank"]

def band(ax, df, x, y, label, color):
    g = df.groupby(x)[y].agg(["mean", "min", "max"]).dropna()
    if len(g):
        ax.plot(g.index, g["mean"], color=color, lw=1, label=label)
        ax.fill_between(g.index, g["min"], g["max"], color=color, alpha=0.15, lw=0)

for dset in sorted(train.dataset.unique()) if len(train) else []:
    tr, ev = train[train.dataset == dset], evals[evals.dataset == dset]
    learners = sorted(tr.learner.unique()); colors = {l: plt.cm.tab20(i % 20) for i, l in enumerate(learners)}
    mets = [m for m in METRICS if m in ev and ev[m].notna().any()]
    fig, axes = plt.subplots(1, 1 + len(mets), figsize=(3.1 * (1 + len(mets)), 2.9))
    for l in learners:
        band(axes[0], tr[tr.learner == l], "step", "loss", l, colors[l])
        for ax, m in zip(axes[1:], mets):
            band(ax, ev[ev.learner == l], "step", m, l, colors[l])
    axes[0].set_xscale("log"); axes[0].set_yscale("symlog", linthresh=1e-2); axes[0].set_title("training loss")
    for ax, m in zip(axes[1:], mets):
        ax.set_xscale("symlog", linthresh=1000); ax.set_title(m + (" (supervised)" if m.startswith("probe") else ""))
        if m != "eff_rank": ax.set_ylim(0, 1.02)
    axes[0].legend(fontsize=5, ncol=1, loc="lower left"); fig.suptitle(dset, fontsize=8, x=0.01, ha="left")
    plt.tight_layout(); plt.show()

## Last evaluation, peaks, convergence

In [ ]:
rows = []
for (dset, learner, seed), ev in evals.groupby(["dataset", "learner", "seed"]) if len(evals) else []:
    ev = ev.sort_values("step"); last = ev.iloc[-1]; tr = train[(train.dataset == dset) & (train.learner == learner) & (train.seed == seed)].sort_values("step")
    row = dict(dataset=dset, learner=learner, seed=seed, step=int(last.step))
    if len(tr) > 10:
        n = max(2, len(tr) // 5)
        row["loss_drop_last_fifth_%"] = 100 * (tr.loss.iloc[-2 * n:-n].mean() - tr.loss.iloc[-n:].mean()) / abs(tr.loss.iloc[-2 * n:-n].mean())
    for m in METRICS:
        if m in ev and ev[m].notna().any():
            row[f"{m} last"] = last[m]; row[f"{m} peak"] = ev[m].max(); row[f"{m} peak step"] = int(ev.loc[ev[m].idxmax(), "step"])
    rows.append(row)
summary = pd.DataFrame(rows)
if len(summary):
    cols = [c for c in summary.columns if c not in ("seed",)]
    display(summary[cols].groupby(["dataset", "learner"]).mean(numeric_only=True).round(3))

## Optional: test-set evaluation of kept checkpoints

Runs with `keep_ckpts` store `ckpt_<step>.pt`. This evaluates them on the test split with the chosen detectors (the same scoring as the benchmark), giving test curves along training. Needs the dataset YAML and the model files; on the cluster or after copying them.

In [ ]:
RUN_CKPT_EVAL = False                     # ✏️
CKPT_DETECTORS = ["hmm", "arhmm", "temporal_excess"]

def checkpoint_curve(run_dir, detectors=CKPT_DETECTORS):
    info = json.load(open(os.path.join(run_dir, "learner.json")))
    bench = Benchmark(info["dataset"], out="/tmp/unused", log=lambda *a: None)
    rows = []
    for ck in sorted(glob.glob(os.path.join(run_dir, "ckpt_*.pt")), key=lambda p: int(p.split("_")[-1][:-3])):
        f = make_learner(info["name"], **info["params"]).load(ck, D=bench.ds.D)
        views, starts = f.represent(bench.test)
        for n in detectors:
            r = score(make_detector(n).detect(views["features"].astype(float), bench.ds.K, seed=0), starts, f.window, bench.test, bench.ds.K, bench.ds.margin)
            rows.append(dict(step=int(ck.split("_")[-1][:-3]), detector=n, **{k: r[k] for k in ("acc", "cp_ap") if k in r}))
    return pd.DataFrame(rows)

if RUN_CKPT_EVAL and len(train):
    for run in sorted(train.run.unique())[:4]:          # ✏️ choose runs
        c = checkpoint_curve(run)
        if len(c):
            display(c.pivot_table(index="step", columns="detector", values="acc").round(3).rename_axis(run.split(os.sep)[-4]))